<a href="https://colab.research.google.com/github/gustavofurini/mestrado/blob/main/busca_a_estrela.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmo Busca A*

In [ ]:
import random
from collections import deque

# Símbolos usados para exibição
SIMBOLOS = {
    "livre": "-",
    "obstaculo": "■",
    "inicio": "A",
    "final": "B",
    "cima": "^",
    "baixo": "v",
    "esquerda": "<",
    "direita": ">"
}

# Variáveis globais
mapa = []
lista_aberta = []
lista_fechada = []
dic_custos = {}
inicio = ()
final = ()

def gerar_mapa(tamanho):
    global mapa
    while True:
        mapa = [[0 if random.random() > 0.3 else 1 for _ in range(tamanho)] for _ in range(tamanho)]
        livres = [(i, j) for i in range(tamanho) for j in range(tamanho) if mapa[i][j] == 0]
        if len(livres) >= 2:
            break

def exibir_mapa_temporario():
    print("\n### MAPA GERADO ###\n")
    for linha in mapa:
        print(" ".join([SIMBOLOS["livre"] if cel == 0 else SIMBOLOS["obstaculo"] for cel in linha]))
    print()

def entrada_de_dados():
    global inicio, final
    print("Digite o tamanho do mapa (n para n x n): ")
    while True:
        try:
            tamanho = int(input("Tamanho: "))
            if tamanho < 2:
                print("Tamanho deve ser maior que 1.")
                continue
            gerar_mapa(tamanho)
            exibir_mapa_temporario()
            break
        except:
            print("Entrada inválida.")

    print("Agora escolha as coordenadas de início e fim (linha coluna)")
    while True:
        try:
            entrada = input("Coordenada inicial: ")
            linha, coluna = map(int, entrada.strip().split())
            if mapa[linha][coluna] == 1:
                print("Esse ponto é um obstáculo. Escolha outro.")
                continue
            inicio = (linha, coluna)
            break
        except:
            print("Entrada inválida.")

    while True:
        try:
            entrada = input("Coordenada final: ")
            linha, coluna = map(int, entrada.strip().split())
            if mapa[linha][coluna] == 1:
                print("Esse ponto é um obstáculo. Escolha outro.")
                continue
            final = (linha, coluna)
            if caminho_existe(inicio, final):
                break
            else:
                print("Não há caminho entre os pontos. Escolha outra coordenada final.")
        except:
            print("Entrada inválida.")

def caminho_existe(inicio_tmp, final_tmp):
    visitados = set()
    fila = deque([inicio_tmp])

    while fila:
        atual = fila.popleft()
        if atual == final_tmp:
            return True
        for viz in encontra_vizinhos(atual, ignorar_fechada=True):
            if viz not in visitados:
                visitados.add(viz)
                fila.append(viz)
    return False

def heuristica_manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def encontra_vizinhos(pos, ignorar_fechada=False):
    x, y = pos
    vizinhos = []
    for dx, dy in [(-1,0), (1,0), (0,-1), (0,1)]:
        nx, ny = x + dx, y + dy
        if 0 <= nx < len(mapa) and 0 <= ny < len(mapa[0]):
            if mapa[nx][ny] != 1 and (ignorar_fechada or (nx, ny) not in lista_fechada):
                vizinhos.append((nx, ny))
    return vizinhos

def calcula_custos(pai, vizinhos):
    for viz in vizinhos:
        g = dic_custos[pai][1] + 1
        h = heuristica_manhattan(viz, final)
        if viz not in dic_custos or g < dic_custos[viz][1]:
            dic_custos[viz] = (viz, g, h, pai)

def ordenar_lista_aberta():
    lista_aberta.sort(key=lambda x: dic_custos[x][1] + dic_custos[x][2])

def recuperar_caminho(no):
    caminho = [no]
    while no != inicio:
        no = dic_custos[no][3]
        caminho.append(no)
    caminho.reverse()
    return caminho

def buscar():
    lista_aberta.clear()
    lista_fechada.clear()
    dic_custos.clear()

    lista_aberta.append(inicio)
    dic_custos[inicio] = (inicio, 0, heuristica_manhattan(inicio, final), None)

    while lista_aberta:
        ordenar_lista_aberta()
        atual = lista_aberta.pop(0)
        lista_fechada.append(atual)

        if atual == final:
            caminho = recuperar_caminho(atual)
            desenhar(caminho)
            return

        vizinhos = encontra_vizinhos(atual)
        calcula_custos(atual, vizinhos)

        for viz in vizinhos:
            if viz not in lista_aberta:
                lista_aberta.append(viz)

    print("Caminho não encontrado.")

def desenhar(caminho):
    print("\n### PERCURSO ###\n")
    grade = [[SIMBOLOS["livre"] if cel == 0 else SIMBOLOS["obstaculo"] for cel in linha] for linha in mapa]

    for i in range(1, len(caminho)):
        x0, y0 = caminho[i - 1]
        x1, y1 = caminho[i]
        if caminho[i] == final:
            continue
        if x1 < x0:
            grade[x1][y1] = SIMBOLOS["cima"]
        elif x1 > x0:
            grade[x1][y1] = SIMBOLOS["baixo"]
        elif y1 < y0:
            grade[x1][y1] = SIMBOLOS["esquerda"]
        elif y1 > y0:
            grade[x1][y1] = SIMBOLOS["direita"]

    ix, iy = inicio
    fx, fy = final
    grade[ix][iy] = SIMBOLOS["inicio"]
    grade[fx][fy] = SIMBOLOS["final"]

    for linha in grade:
        print(" ".join(linha))
    print()

def main():
    while True:
        entrada_de_dados()

        if inicio == final:
            print("Você já está no destino.")
        else:
            buscar()

        repetir = input("Deseja tentar novamente? (s para sim): ").strip().lower()
        if repetir != 's':
            print("Encerrando...")
            break

if __name__ == "__main__":
    main()


Digite o tamanho do mapa (n para n x n): 
Tamanho: 7

### MAPA GERADO ###

■ - - - ■ - -
- - - - - - ■
- - ■ - - - -
■ ■ ■ - - - -
- - - - - - -
- - - ■ - - ■
■ ■ - - ■ - ■

Agora escolha as coordenadas de início e fim (linha coluna)
Coordenada inicial: 0 6
Coordenada final: 6 3

### PERCURSO ###

■ - - - ■ < A
- - - - - v ■
- - ■ - - v -
■ ■ ■ - - v -
- - < < < v -
- - v ■ - - ■
■ ■ v B ■ - ■

Deseja tentar novamente? (s para sim): s
Digite o tamanho do mapa (n para n x n): 
Tamanho: 15

### MAPA GERADO ###

- ■ ■ - ■ - ■ - - - - - - - -
- ■ ■ - ■ - ■ - - - - ■ ■ ■ -
- - - - - - ■ ■ ■ - - - - - -
■ ■ ■ - ■ ■ - - ■ - - - - - -
■ ■ - - ■ - ■ ■ ■ - ■ - - - -
- - ■ - - ■ ■ - - - - ■ ■ - ■
■ - - ■ - - - ■ - - ■ - - - ■
■ - - - ■ - - ■ - ■ - - ■ - -
- - - - - - ■ - - - ■ ■ ■ ■ -
- - ■ - - ■ - - - ■ - - ■ ■ -
- - - - ■ ■ - - - - - - - - -
- - ■ ■ ■ ■ - ■ - - - - ■ ■ ■
- ■ - - - ■ - - - ■ - - - - -
- - - - ■ - ■ - ■ - - - - - ■
■ - ■ ■ ■ - ■ - ■ ■ ■ - - - -

Agora escolha as coordenadas de in

# Definição da heurística de Manhattan

A heurística de Manhattan calcula a distância entre dois pontos em uma grade, somando as diferenças absolutas entre suas coordenadas horizontais e verticais. É usada em algoritmos como o A* para estimar o custo até o destino. Ela assume que o movimento só pode ser feito em linhas retas (cima, baixo, esquerda e direita). Essa heurística é admissível, ou seja, nunca superestima o custo real. Por isso, garante bons resultados em caminhos ortogonais.